# Toko Bandung Pipeline

toko-raw -> toko-processed -> toko-curated

## Setup

### Import Libraries

In [116]:
import psycopg2
from dotenv import load_dotenv
import os
from pathlib import Path
import pandas as pd
import boto3
from botocore.config import Config
import sys
import hashlib
from pyspark.sql import SparkSession

### Lokasi python kernel

In [107]:
print(sys.executable)

C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\.venv\Scripts\python.exe


### Root folder

In [108]:
BASE_DIR = Path(os.getcwd()).resolve()
load_dotenv(BASE_DIR / ".env")

True

## PostgresSql Exploration

### Membuat koneksi MinIO

In [27]:
# membuat koneksi ke minio
s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9007",
    aws_access_key_id=os.getenv("MINIO_ROOT_USER"),
    aws_secret_access_key=os.getenv("MINIO_ROOT_PASSWORD"),
    region_name="us-east-1",
     config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path"}
    )
)


print("Koneksi boto3 ke MinIO berhasil!")

Koneksi boto3 ke MinIO berhasil!


### Membuat koneksi postgres

In [28]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD")
)

print("Berhasil terhubung ke PostgreSQL!")

Berhasil terhubung ke PostgreSQL!


### Memeriksa koneksi dan membaca sample data

In [109]:
import pandas as pd

query = """
SELECT *
FROM orders
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)

df.head()

C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\1855603530.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,order_id,customer_id,product_id,qty,unit_price,status,order_ts,updated_at
0,912,C0378,P005,4,50000,paid,2026-10-20 00:08:00,2026-10-20 00:10:00
1,1924,C0347,P030,1,120000,shipped,2026-10-20 00:02:00,2026-10-20 00:31:00
2,540,C0029,P035,4,250000,paid,2026-10-20 00:17:00,2026-10-20 00:45:00
3,1768,C0239,P005,2,120000,paid,2026-10-20 00:45:00,2026-10-20 00:58:00
4,1482,C0258,P025,3,120000,paid,2026-10-20 00:48:00,2026-10-20 01:50:00


### Periksa jumlah data dan range kolom update_at

In [111]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(updated_at) AS earliest_update,
    MAX(updated_at) AS latest_update
FROM orders;
"""

source_stats = pd.read_sql_query(query, conn)
source_stats

print("Total rows:", source_stats.loc[0, "total_rows"])
print("Earliest update:", source_stats.loc[0, "earliest_update"])
print("Latest update:", source_stats.loc[0, "latest_update"])

Total rows: 2000
Earliest update: 2026-10-20 00:10:00
Latest update: 2026-10-27 03:28:00


C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\2453500390.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  source_stats = pd.read_sql_query(query, conn)


In [112]:
query = """
SELECT *
FROM orders
ORDER BY updated_at, order_id;
"""

df = pd.read_sql_query(query, conn)

print("Total data berhasil di-extract:", len(df))
print("Jumlah kolom:", len(df.columns))

df.head()

Total data berhasil di-extract: 2000
Jumlah kolom: 8


C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\2669839331.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,order_id,customer_id,product_id,qty,unit_price,status,order_ts,updated_at
0,912,C0378,P005,4,50000,paid,2026-10-20 00:08:00,2026-10-20 00:10:00
1,1924,C0347,P030,1,120000,shipped,2026-10-20 00:02:00,2026-10-20 00:31:00
2,540,C0029,P035,4,250000,paid,2026-10-20 00:17:00,2026-10-20 00:45:00
3,1768,C0239,P005,2,120000,paid,2026-10-20 00:45:00,2026-10-20 00:58:00
4,1482,C0258,P025,3,120000,paid,2026-10-20 00:48:00,2026-10-20 01:50:00


### Memeriksa kandidat watermark

In [18]:
batch_max_updated_at = df["updated_at"].max()

print("Jumlah data batch:", len(df))
print("Watermark kandidat:", batch_max_updated_at)

Jumlah data batch: 2000
Watermark kandidat: 2026-10-27 03:28:00


## 1. Source to raw

data yang sudah masuk kedalam database postgres, di extract kemudian diubah kembali menjadi file csv lalu diload ke dalam Minio Bucket (toko-raw)

In [40]:
import hashlib
import json
from datetime import datetime, timezone

# Ubah DataFrame menjadi CSV dalam memory
csv_data = df.to_csv(index=False, lineterminator="\n").encode("utf-8")

# Hitung SHA-256
raw_checksum = hashlib.sha256(csv_data).hexdigest()

# Tentukan lokasi object di MinIO
raw_key = "orders/initial_load/orders.csv"

# Upload ke MinIO
s3.put_object(
    Bucket="toko-raw",
    Key=raw_key,
    Body=csv_data,
    ContentType="text/csv"
)

print("Raw data berhasil ditulis ke MinIO!")
print("Object key:", raw_key)
print("Jumlah baris:", len(df))
print("Ukuran file:", len(csv_data), "bytes")
print("SHA-256:", raw_checksum)

Raw data berhasil ditulis ke MinIO!
Object key: orders/initial_load/orders.csv
Jumlah baris: 2000
Ukuran file: 140902 bytes
SHA-256: 54f2be84281fdd350c0d2c41df03f154f75f3c68a9d7da3d30479d0a4b5d05ab


In [37]:
# Periksa tipe data DataFrame
print(df.dtypes)

# Periksa ukuran CSV hasil serialisasi
print("Ukuran CSV hasil Pandas:", len(csv_data))

# Periksa jumlah baris
print("Jumlah baris DataFrame:", len(df))

order_id                  str
customer_id               str
product_id                str
qty                     int64
unit_price              int64
status                    str
order_ts       datetime64[us]
updated_at     datetime64[us]
dtype: object
Ukuran CSV hasil Pandas: 140902
Jumlah baris DataFrame: 2000


In [42]:
# Hitung jumlah line ending
print("Jumlah LF:", csv_data.count(b"\n"))
print("Jumlah CRLF:", csv_data.count(b"\r\n"))

# Tampilkan beberapa byte pertama
print("Byte awal CSV:", repr(csv_data[:200]))

Jumlah LF: 2001
Jumlah CRLF: 0
Byte awal CSV: b'order_id,customer_id,product_id,qty,unit_price,status,order_ts,updated_at\n912,C0378,P005,4,50000,paid,2026-10-20 00:08:00,2026-10-20 00:10:00\n1924,C0347,P030,1,120000,shipped,2026-10-20 00:02:00,2026-'


In [41]:
csv_path = BASE_DIR / "data" / "source" / "orders_seed.csv"

with open(csv_path, "rb") as file:
    original_bytes = file.read()

print("Ukuran file asli:", len(original_bytes))
print("Jumlah LF:", original_bytes.count(b"\n"))
print("Jumlah CRLF:", original_bytes.count(b"\r\n"))

Ukuran file asli: 140902
Jumlah LF: 2001
Jumlah CRLF: 0


### Verifikasi data di MinIO

In [44]:
# Tahap 1 — Verifikasi data di MinIO
# Ambil kembali file dari MinIO
response = s3.get_object(
    Bucket="toko-raw",
    Key=raw_key
)

# Baca isi file
raw_from_minio = response["Body"].read()

# Hitung SHA-256 dari file yang tersimpan
minio_checksum = hashlib.sha256(raw_from_minio).hexdigest()

# Hitung jumlah baris data
import io
import pandas as pd

verified_df = pd.read_csv(io.BytesIO(raw_from_minio))

print("=== HASIL VERIFIKASI RAW ===")
print("Jumlah baris:", len(verified_df))
print("Jumlah kolom:", len(verified_df.columns))
print("SHA-256 saat upload:", raw_checksum)
print("SHA-256 dari MinIO:", minio_checksum)
print("Checksum cocok:", raw_checksum == minio_checksum)
print("Row count cocok:", len(verified_df) == len(df))

=== HASIL VERIFIKASI RAW ===
Jumlah baris: 2000
Jumlah kolom: 8
SHA-256 saat upload: 54f2be84281fdd350c0d2c41df03f154f75f3c68a9d7da3d30479d0a4b5d05ab
SHA-256 dari MinIO: 54f2be84281fdd350c0d2c41df03f154f75f3c68a9d7da3d30479d0a4b5d05ab
Checksum cocok: True
Row count cocok: True


In [45]:
# simpan watermark
import json
from datetime import datetime, timezone

watermark_data = {
    "table": "orders",
    "last_updated_at": batch_max_updated_at.isoformat(),
    "row_count": len(df),
    "raw_key": raw_key,
    "sha256": raw_checksum,
    "updated_at": datetime.now(timezone.utc).isoformat()
}

s3.put_object(
    Bucket="toko-raw",
    Key="checkpoints/orders_watermark.json",
    Body=json.dumps(watermark_data, indent=4).encode("utf-8"),
    ContentType="application/json"
)

print("Watermark berhasil disimpan!")
print(watermark_data)

Watermark berhasil disimpan!
{'table': 'orders', 'last_updated_at': '2026-10-27T03:28:00', 'row_count': 2000, 'raw_key': 'orders/initial_load/orders.csv', 'sha256': '54f2be84281fdd350c0d2c41df03f154f75f3c68a9d7da3d30479d0a4b5d05ab', 'updated_at': '2026-11-20T10:42:38.123755+00:00'}


## 2. Raw - Processed

data raw yang sudah masuk ke dalam minio toko-raw, ditarik kembali untuk diproses dan masuk ke layer selanjutnya toko-procesed

### Membuat sparksession

In [117]:
# konfigurasi spark session
spark = (
    SparkSession.builder
    .appName("TokoBandung-DataProcessing")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [118]:
#  Ambil data dari MinIO Raw
response = s3.get_object(
    Bucket="toko-raw",
    Key=raw_key
)

raw_from_minio = response["Body"].read()
raw_local_path = BASE_DIR / "data" / "source" / "raw_from_minio.csv"

with open(raw_local_path, "wb") as f:
    f.write(raw_from_minio)

# Baca menggunakan Spark
orders_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(raw_local_path))
)

print("Jumlah data:", orders_spark.count())
orders_spark.show(5)

Jumlah data: 2000
+--------+-----------+----------+---+----------+-------+-------------------+-------------------+
|order_id|customer_id|product_id|qty|unit_price| status|           order_ts|         updated_at|
+--------+-----------+----------+---+----------+-------+-------------------+-------------------+
|     912|      C0378|      P005|  4|     50000|   paid|2026-10-20 00:08:00|2026-10-20 00:10:00|
|    1924|      C0347|      P030|  1|    120000|shipped|2026-10-20 00:02:00|2026-10-20 00:31:00|
|     540|      C0029|      P035|  4|    250000|   paid|2026-10-20 00:17:00|2026-10-20 00:45:00|
|    1768|      C0239|      P005|  2|    120000|   paid|2026-10-20 00:45:00|2026-10-20 00:58:00|
|    1482|      C0258|      P025|  3|    120000|   paid|2026-10-20 00:48:00|2026-10-20 01:50:00|
+--------+-----------+----------+---+----------+-------+-------------------+-------------------+
only showing top 5 rows


### Standarisasi dan quality gate

ketentuan 
Kolom	Aturan
order_id	Tidak boleh kosong
product_id	Tidak boleh kosong
qty	Harus lebih dari 0
unit_price	Harus 25.000, 50.000, 75.000, 120.000, atau 250.000
status	Hanya PAID atau CANCELLED
order_ts	Tidak boleh kosong
order_id	Tidak boleh duplikat

In [56]:
# standarisasi status
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#### 1. Standardisasi status ( "PAID", " paid " -> paid)

In [132]:
standardized_df = orders_spark.withColumn(
    "status",
    F.lower(F.trim(F.col("status")))
)

In [133]:
standardized_df.show(5)

+--------+-----------+----------+---+----------+-------+-------------------+-------------------+
|order_id|customer_id|product_id|qty|unit_price| status|           order_ts|         updated_at|
+--------+-----------+----------+---+----------+-------+-------------------+-------------------+
|     912|      C0378|      P005|  4|     50000|   paid|2026-10-20 00:08:00|2026-10-20 00:10:00|
|    1924|      C0347|      P030|  1|    120000|shipped|2026-10-20 00:02:00|2026-10-20 00:31:00|
|     540|      C0029|      P035|  4|    250000|   paid|2026-10-20 00:17:00|2026-10-20 00:45:00|
|    1768|      C0239|      P005|  2|    120000|   paid|2026-10-20 00:45:00|2026-10-20 00:58:00|
|    1482|      C0258|      P025|  3|    120000|   paid|2026-10-20 00:48:00|2026-10-20 01:50:00|
+--------+-----------+----------+---+----------+-------+-------------------+-------------------+
only showing top 5 rows


#### 2. Identifikasi duplikasi order_id

In [134]:
# - kelompokan berdasarkan order_id, lalu urutkan berdasarkan update_at, dan order_ts
window_spec = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").desc(),
    F.col("order_ts").desc()
)

# tambahkam row number untuk deteksi duplikat
checked_df = standardized_df.withColumn(
    "_row_number",
    F.row_number().over(window_spec)
)

# lihat sample data
checked_df.show(10)

total_rows = checked_df.count()
unique_orders = checked_df.select("order_id").distinct().count()
print(f"Total rows: {total_rows}")
print(f"Unique order_id: {unique_orders}")
print(f"Duplicate rows: {total_rows - unique_orders}")

+--------+-----------+----------+---+----------+---------+-------------------+-------------------+-----------+
|order_id|customer_id|product_id|qty|unit_price|   status|           order_ts|         updated_at|_row_number|
+--------+-----------+----------+---+----------+---------+-------------------+-------------------+-----------+
|       1|      C0306|      P012|  2|     50000|     paid|2026-10-21 06:50:00|2026-10-21 10:01:00|          1|
|       2|      C0110|      P010|  4|     50000|     paid|2026-10-25 06:43:00|2026-10-25 09:59:00|          1|
|       3|      C0015|      P027|  2|     25000|     paid|2026-10-21 11:47:00|2026-10-21 14:23:00|          1|
|       4|      C0162|      P009|  2|    250000|cancelled|2026-10-21 01:56:00|2026-10-21 03:44:00|          1|
|       5|      C0409|      P018|  2|    120000|  shipped|2026-10-25 21:55:00|2026-10-26 00:52:00|          1|
|       6|       NULL|      P019|  3|     25000|     paid|2026-10-21 11:53:00|2026-10-21 12:45:00|          1|
|

#### 3. Buat alasan penolakan

In [135]:
checked_df = checked_df.withColumn(
    "reject_reason",
    F.concat_ws(
        "; ",
        F.when(
            F.col("order_id").isNull() | (F.trim(F.col("order_id")) == ""),
            F.lit("Missing order_id")
        ),
        F.when(
            F.col("product_id").isNull() | (F.trim(F.col("product_id")) == ""),
            F.lit("Missing product_id")
        ),
        F.when(
            F.col("qty").isNull() | (F.col("qty") <= 0),
            F.lit("Invalid quantity")
        ),
        F.when(
            F.col("unit_price").isNull() |
            (~F.col("unit_price").isin(25000, 50000, 75000, 120000, 250000)),
            F.lit("Invalid unit_price")
        ),
        F.when(
            F.col("status").isNull() |
            (~F.col("status").isin("paid", "cancelled")),
            F.lit("Unsupported status")
        ),
        F.when(
            F.col("order_ts").isNull(),
            F.lit("Missing order_ts")
        ),
        F.when(
            F.col("_row_number") > 1,
            F.lit("Duplicate order_id")
        )
    )
)

In [136]:
checked_df.show(10)

+--------+-----------+----------+---+----------+---------+-------------------+-------------------+-----------+------------------+
|order_id|customer_id|product_id|qty|unit_price|   status|           order_ts|         updated_at|_row_number|     reject_reason|
+--------+-----------+----------+---+----------+---------+-------------------+-------------------+-----------+------------------+
|       1|      C0306|      P012|  2|     50000|     paid|2026-10-21 06:50:00|2026-10-21 10:01:00|          1|                  |
|       2|      C0110|      P010|  4|     50000|     paid|2026-10-25 06:43:00|2026-10-25 09:59:00|          1|                  |
|       3|      C0015|      P027|  2|     25000|     paid|2026-10-21 11:47:00|2026-10-21 14:23:00|          1|                  |
|       4|      C0162|      P009|  2|    250000|cancelled|2026-10-21 01:56:00|2026-10-21 03:44:00|          1|                  |
|       5|      C0409|      P018|  2|    120000|  shipped|2026-10-25 21:55:00|2026-10-26 0

#### 4. Pisahkan data valid dan quarantine

In [137]:
valid_df = (
    checked_df
    .filter(F.col("reject_reason") == "")
    .drop("_row_number", "reject_reason")
)

valid_df.show(5)

+--------+-----------+----------+---+----------+---------+-------------------+-------------------+
|order_id|customer_id|product_id|qty|unit_price|   status|           order_ts|         updated_at|
+--------+-----------+----------+---+----------+---------+-------------------+-------------------+
|       1|      C0306|      P012|  2|     50000|     paid|2026-10-21 06:50:00|2026-10-21 10:01:00|
|       2|      C0110|      P010|  4|     50000|     paid|2026-10-25 06:43:00|2026-10-25 09:59:00|
|       3|      C0015|      P027|  2|     25000|     paid|2026-10-21 11:47:00|2026-10-21 14:23:00|
|       4|      C0162|      P009|  2|    250000|cancelled|2026-10-21 01:56:00|2026-10-21 03:44:00|
|       6|       NULL|      P019|  3|     25000|     paid|2026-10-21 11:53:00|2026-10-21 12:45:00|
+--------+-----------+----------+---+----------+---------+-------------------+-------------------+
only showing top 5 rows


In [139]:
quarantine_df = (
    checked_df
    .filter(F.col("reject_reason") != "")
    .drop("_row_number")
)

quarantine_df.show(5)

+--------+-----------+----------+---+----------+-------+-------------------+-------------------+------------------+
|order_id|customer_id|product_id|qty|unit_price| status|           order_ts|         updated_at|     reject_reason|
+--------+-----------+----------+---+----------+-------+-------------------+-------------------+------------------+
|       5|      C0409|      P018|  2|    120000|shipped|2026-10-25 21:55:00|2026-10-26 00:52:00|Unsupported status|
|      10|      C0098|      P035|  4|     25000|shipped|2026-10-21 23:04:00|2026-10-22 00:24:00|Unsupported status|
|      12|      C0077|      P032|  2|    250000|shipped|2026-10-21 05:01:00|2026-10-21 07:09:00|Unsupported status|
|      15|      C0226|      P009|  1|    120000|shipped|2026-10-24 15:10:00|2026-10-24 18:06:00|Unsupported status|
|      18|      C0088|      P034|  1|    120000|shipped|2026-10-21 17:45:00|2026-10-21 21:00:00|Unsupported status|
+--------+-----------+----------+---+----------+-------+----------------

#### 5. Tampilkan hasil

In [89]:
total_rows = orders_spark.count()
valid_count = valid_df.count()
rejected_count = quarantine_df.count()

print("=== QUALITY GATE REPORT ===")
print("Total data masuk :", total_rows)
print("Data lolos       :", valid_count)
print("Data ditolak     :", rejected_count)
print("Reconciliation   :", valid_count + rejected_count == total_rows)

=== QUALITY GATE REPORT ===
Total data masuk : 2000
Data lolos       : 1269
Data ditolak     : 731
Reconciliation   : True


In [86]:
print("\n=== ALASAN PENOLAKAN ===")
quarantine_df.groupBy("reject_reason").count().show(truncate=False)


=== ALASAN PENOLAKAN ===
+------------------------------------+-----+
|reject_reason                       |count|
+------------------------------------+-----+
|Unsupported status                  |721  |
|Invalid quantity                    |7    |
|Invalid quantity; Unsupported status|3    |
+------------------------------------+-----+



### Simpan ke MinIO Processed

In [94]:
# 1. Buat ID untuk setiap proses
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# 2. Tentukan lokasi penyimpanan sementara
processed_local = BASE_DIR / "outputs" / "processed" / run_id
quarantine_local = BASE_DIR / "outputs" / "quarantine" / run_id

In [95]:
# 3. Simpan data valid sebagai Parquet
(
    valid_df.coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(processed_local))
)

# 4. Simpan data rejected sebagai Parquet
(
    quarantine_df.coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(quarantine_local))
)

print("Parquet berhasil dibuat secara lokal.")

Parquet berhasil dibuat secara lokal.


In [97]:
# fungsi upload
def upload_parquet_folder(local_folder, bucket, prefix):
    local_folder = Path(local_folder)

    for file_path in local_folder.rglob("*"):
        if file_path.is_file() and not file_path.name.startswith("."):
            relative_path = file_path.relative_to(local_folder)
            object_key = f"{prefix}/{relative_path.as_posix()}"

            s3.upload_file(
                str(file_path),
                bucket,
                object_key
            )

            print("Uploaded:", object_key)

In [98]:
# Upload data valid
upload_parquet_folder(
    processed_local,
    "toko-processed",
    f"orders/run_id={run_id}"
)

Uploaded: orders/run_id=20261120_201432/part-00000-00c2101d-a23e-41d8-b4a1-3479bcb7c902-c000.snappy.parquet
Uploaded: orders/run_id=20261120_201432/_SUCCESS


In [99]:
# Upload data quarantine
upload_parquet_folder(
    quarantine_local,
    "toko-processed",
    f"quarantine/run_id={run_id}"
)

Uploaded: quarantine/run_id=20261120_201432/part-00000-6b94ecf1-d3c9-4f85-943f-f6447ae9dd38-c000.snappy.parquet
Uploaded: quarantine/run_id=20261120_201432/_SUCCESS


In [140]:
# verifikasi 
response = s3.list_objects_v2(
    Bucket="toko-processed",
    Prefix=f""
)

print("=== OBJECT DI TOKO-PROCESSED ===")

for obj in response.get("Contents", []):
    print(obj["Key"])

=== OBJECT DI TOKO-PROCESSED ===
orders/run_id=20261120_201432/_SUCCESS
orders/run_id=20261120_201432/part-00000-00c2101d-a23e-41d8-b4a1-3479bcb7c902-c000.snappy.parquet
quarantine/run_id=20261120_201432/_SUCCESS
quarantine/run_id=20261120_201432/part-00000-6b94ecf1-d3c9-4f85-943f-f6447ae9dd38-c000.snappy.parquet


## 3. Processed to Curated

data dari MinIo bucket toko-processed di ambil kembali lalu buat tabel daily_sales dan product_sales

1. daily_sales: ringkasan penjualan per tanggal.
2. product_sales: ringkasan penjualan per produk.

### 1. Baca data dari MinIo toko-processed

In [151]:
# Menentukan lokasi data di MinIO
processed_prefix = f"orders/run_id={run_id}/"
print(processed_prefix)

# tempat menyimpan data dari toko-procesed secara lokal
processed_from_minio = BASE_DIR / "outputs" / "from_minio" / run_id
processed_from_minio.mkdir(
    parents=True,
    exist_ok=True
)

orders/run_id=20261120_201432/


In [152]:
# Meminta daftar object dari MinIO
response = s3.list_objects_v2(
    Bucket="toko-processed",
    Prefix=processed_prefix
)

In [157]:
# simpan ke lokal 
for obj in response.get("Contents", []):
    key = obj["Key"]

    if key.endswith(".parquet"):
        filename = key.split("/")[-1]

        s3.download_file(
            "toko-processed",
            key,
            str(processed_from_minio / filename)
        )

In [156]:
# Baca kembali dengan Spark
processed_df = spark.read.parquet(str(processed_from_minio))

print("Jumlah data Processed:", processed_df.count())
processed_df.show(5)

Jumlah data Processed: 1269
+--------+-----------+----------+---+----------+---------+-------------------+-------------------+
|order_id|customer_id|product_id|qty|unit_price|   status|           order_ts|         updated_at|
+--------+-----------+----------+---+----------+---------+-------------------+-------------------+
|       1|      C0306|      P012|  2|     50000|     paid|2026-10-21 06:50:00|2026-10-21 10:01:00|
|       2|      C0110|      P010|  4|     50000|     paid|2026-10-25 06:43:00|2026-10-25 09:59:00|
|       3|      C0015|      P027|  2|     25000|     paid|2026-10-21 11:47:00|2026-10-21 14:23:00|
|       4|      C0162|      P009|  2|    250000|cancelled|2026-10-21 01:56:00|2026-10-21 03:44:00|
|       6|       NULL|      P019|  3|     25000|     paid|2026-10-21 11:53:00|2026-10-21 12:45:00|
+--------+-----------+----------+---+----------+---------+-------------------+-------------------+
only showing top 5 rows


### 2. Hitung daily sales dan product sales

In [162]:
# Hanya transaksi yang bukan CANCELLED
sales_df = (
    processed_df
    .filter(F.col("status") != "cancelled")
    .withColumn("dt", F.to_date("order_ts"))
    .withColumn(
        "revenue",
        F.col("qty") * F.col("unit_price")
    )
)

sales_df.show(5)
sales_df.count()

+--------+-----------+----------+---+----------+------+-------------------+-------------------+----------+-------+
|order_id|customer_id|product_id|qty|unit_price|status|           order_ts|         updated_at|        dt|revenue|
+--------+-----------+----------+---+----------+------+-------------------+-------------------+----------+-------+
|       1|      C0306|      P012|  2|     50000|  paid|2026-10-21 06:50:00|2026-10-21 10:01:00|2026-10-21| 100000|
|       2|      C0110|      P010|  4|     50000|  paid|2026-10-25 06:43:00|2026-10-25 09:59:00|2026-10-25| 200000|
|       3|      C0015|      P027|  2|     25000|  paid|2026-10-21 11:47:00|2026-10-21 14:23:00|2026-10-21|  50000|
|       6|       NULL|      P019|  3|     25000|  paid|2026-10-21 11:53:00|2026-10-21 12:45:00|2026-10-21|  75000|
|       8|      C0220|      P032|  3|     50000|  paid|2026-10-22 07:11:00|2026-10-22 08:46:00|2026-10-22| 150000|
+--------+-----------+----------+---+----------+------+-------------------+-----

1062

In [167]:
# 1. Penjualan harian
daily_sales = (
    sales_df
    .groupBy("dt")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("qty").alias("item_count"),
        F.sum("revenue").alias("revenue")
    )
    .orderBy("dt")
)

daily_sales.show()
daily_sales.count()

+----------+-----------+----------+--------+
|        dt|order_count|item_count| revenue|
+----------+-----------+----------+--------+
|2026-10-20|        134|       338|34600000|
|2026-10-21|        158|       378|39640000|
|2026-10-22|        154|       388|42205000|
|2026-10-23|        156|       386|40790000|
|2026-10-24|        159|       386|41960000|
|2026-10-25|        142|       358|33930000|
|2026-10-26|        159|       389|40945000|
+----------+-----------+----------+--------+



7

In [174]:
# 2. Penjualan per produk
product_sales = (
    sales_df
    .groupBy("product_id")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("qty").alias("item_count"),
        F.sum("revenue").alias("revenue")
    )
    .orderBy("product_id")
)

product_sales.show(40)
product_sales.count()

+----------+-----------+----------+--------+
|product_id|order_count|item_count| revenue|
+----------+-----------+----------+--------+
|      P001|         25|        57| 5325000|
|      P002|         32|        82| 8710000|
|      P003|         30|        72| 4950000|
|      P004|         24|        60| 6540000|
|      P005|         30|        80| 8615000|
|      P006|         19|        47| 5245000|
|      P007|         27|        67| 7085000|
|      P008|         22|        63| 8535000|
|      P009|         35|        85| 8475000|
|      P010|         29|        73| 9495000|
|      P011|         24|        66| 7810000|
|      P012|         33|        82| 7760000|
|      P013|         23|        58| 4355000|
|      P014|         31|        67| 7015000|
|      P015|         25|        52| 4605000|
|      P016|         26|        75| 6240000|
|      P017|         21|        52| 7205000|
|      P018|         31|        90|10235000|
|      P019|         21|        52| 4090000|
|      P02

40

### 3. Simpan ke MinIO bucket toko-curated dengan format parquet

In [170]:
# lokasi penyimpanan lokal
curated_local = BASE_DIR / "outputs" / "curated" / run_id

daily_local = curated_local / "daily_sales"
product_local = curated_local / "product_sales"

In [171]:
# Simpan daily_sales dengan partition dt
(
    daily_sales
    .write
    .mode("overwrite")
    .partitionBy("dt")
    .parquet(str(daily_local))
)

# Simpan product_sales
(
    product_sales
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(product_local))
)

print("Curated Parquet berhasil dibuat!")

Curated Parquet berhasil dibuat!


In [172]:
upload_parquet_folder(
    daily_local,
    "toko-curated",
    "daily_sales"
)

upload_parquet_folder(
    product_local,
    "toko-curated",
    "product_sales"
)

Uploaded: daily_sales/_SUCCESS
Uploaded: daily_sales/dt=2026-10-20/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
Uploaded: daily_sales/dt=2026-10-21/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
Uploaded: daily_sales/dt=2026-10-22/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
Uploaded: daily_sales/dt=2026-10-23/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
Uploaded: daily_sales/dt=2026-10-24/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
Uploaded: daily_sales/dt=2026-10-25/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
Uploaded: daily_sales/dt=2026-10-26/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
Uploaded: product_sales/part-00000-bca6a535-7a33-4f2d-8249-23da27d299b9-c000.snappy.parquet
Uploaded: product_sales/_SUCCESS


### 4. Periksa hasilnya apakah sudah masuk ke MInIO

In [173]:
response = s3.list_objects_v2(
    Bucket="toko-curated"
)

print("=== OBJECT DI TOKO-CURATED ===")

for obj in response.get("Contents", []):
    print(obj["Key"])

=== OBJECT DI TOKO-CURATED ===
daily_sales/_SUCCESS
daily_sales/dt=2026-10-20/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
daily_sales/dt=2026-10-21/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
daily_sales/dt=2026-10-22/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
daily_sales/dt=2026-10-23/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
daily_sales/dt=2026-10-24/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
daily_sales/dt=2026-10-25/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
daily_sales/dt=2026-10-26/part-00000-6a97f9b3-cadc-4d4a-8185-b199e4b55576.c000.snappy.parquet
product_sales/_SUCCESS
product_sales/part-00000-bca6a535-7a33-4f2d-8249-23da27d299b9-c000.snappy.parquet


### 5. Reconciliation kedua tabel

In [176]:
# Total dari daily_sales
daily_totals = daily_sales.agg(
    F.sum("order_count").alias("orders"),
    F.sum("item_count").alias("items"),
    F.sum("revenue").alias("revenue")
).first()

# Total dari product_sales
product_totals = product_sales.agg(
    F.sum("order_count").alias("orders"),
    F.sum("item_count").alias("items"),
    F.sum("revenue").alias("revenue")
).first()

print("=== RECONCILIATION CURATED ===")
print("Daily sales   :", daily_totals)
print("Product sales :", product_totals)

print("Order match  :", daily_totals["orders"] == product_totals["orders"])
print("Item match   :", daily_totals["items"] == product_totals["items"])
print("Revenue match:", daily_totals["revenue"] == product_totals["revenue"])

=== RECONCILIATION CURATED ===
Daily sales   : Row(orders=1062, items=2623, revenue=274070000)
Product sales : Row(orders=1062, items=2623, revenue=274070000)
Order match  : True
Item match   : True
Revenue match: True


## Reconciliation data sumber dengan data akhir pipeline

In [179]:
source_reconciliation_query = """
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(qty) AS total_items,
    SUM(qty * unit_price) AS total_revenue
FROM orders
WHERE LOWER(TRIM(status)) = 'paid'
    AND order_id IS NOT NULL
    AND TRIM(order_id) <> ''
    AND product_id IS NOT NULL
    AND TRIM(product_id) <> ''
    AND qty > 0
    AND unit_price IN (25000, 50000, 75000, 120000, 250000)
    AND order_ts IS NOT NULL;
"""

source_metrics = pd.read_sql_query(
    source_reconciliation_query,
    conn
)

source_metrics

C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\2985420646.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  source_metrics = pd.read_sql_query(


,total_orders,total_items,total_revenue
0,1062,2623,274070000


In [180]:
source_orders = int(source_metrics.iloc[0]["total_orders"])
source_items = int(source_metrics.iloc[0]["total_items"])
source_revenue = int(source_metrics.iloc[0]["total_revenue"])

curated_metrics = sales_df.agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("qty").alias("total_items"),
    F.sum("revenue").alias("total_revenue")
).collect()[0]

curated_orders = int(curated_metrics["total_orders"])
curated_items = int(curated_metrics["total_items"])
curated_revenue = int(curated_metrics["total_revenue"])

reconciliation = pd.DataFrame({
    "metric": ["total_orders", "total_items", "total_revenue"],
    "source_postgres": [source_orders, source_items, source_revenue],
    "curated_lake": [curated_orders, curated_items, curated_revenue],
})

reconciliation["match"] = (
    reconciliation["source_postgres"]
    == reconciliation["curated_lake"]
)

reconciliation

,metric,source_postgres,curated_lake,match
0,total_orders,1062,1062,True
1,total_items,2623,2623,True
2,total_revenue,274070000,274070000,True


## Incremental Extration

In [181]:
import json
from datetime import datetime
from io import BytesIO

In [190]:
watermark_key = "checkpoints/orders_watermark.json"

# 1. Baca watermark terakhir
watermark_obj = s3.get_object(
    Bucket="toko-raw",
    Key=watermark_key
)

watermark_data = json.loads(
    watermark_obj["Body"].read().decode("utf-8")
)

last_watermark = watermark_data["last_updated_at"]

print("Last watermark:", last_watermark)

# 2. Extract data incremental dari PostgreSQL
incremental_query = """
SELECT *
FROM orders
WHERE updated_at >= %s
ORDER BY updated_at, order_id;
"""

incremental_df = pd.read_sql_query(
    incremental_query,
    conn,
    params=(last_watermark,)
)

print("Incremental rows:", len(incremental_df))


Last watermark: 2026-10-27 03:28:00
Incremental rows: 1


C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\1738870248.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  incremental_df = pd.read_sql_query(


In [191]:
response = s3.list_objects_v2(
    Bucket="toko-raw"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

checkpoints/orders_watermark.json
orders/incremental/from=2026-10-27T03:28:00/to=2026-10-27 03:28:00/orders.csv
orders/initial_load/orders.csv
orders/source/orders_seed.csv
orders/source/orders_seed.metadata.json


## Upload batch dan update watermark

In [192]:
if not incremental_df.empty:

    # Ambil watermark baru
    new_watermark = str(
        incremental_df["updated_at"].max()
    )

    # Buat CSV di memory
    csv_buffer = BytesIO()

    incremental_df.to_csv(
        csv_buffer,
        index=False,
        lineterminator="\n"
    )

    csv_bytes = csv_buffer.getvalue()

    # Batch key berdasarkan rentang watermark
    batch_key = (
        f"orders/incremental/"
        f"from={last_watermark}/"
        f"to={new_watermark}/orders.csv"
    )

    # 3. Upload batch ke Raw MinIO
    s3.put_object(
        Bucket="toko-raw",
        Key=batch_key,
        Body=csv_bytes,
        ContentType="text/csv"
    )

    print("Uploaded batch:", batch_key)
    print("Rows uploaded:", len(incremental_df))

    # 4. Update watermark setelah upload berhasil
    new_watermark_data = {
        "table": "orders",
        "last_updated_at": new_watermark,
        "row_count": watermark_data["row_count"],
        "raw_key": batch_key,
        "updated_at": datetime.now().astimezone().isoformat()
    }

    s3.put_object(
        Bucket="toko-raw",
        Key=watermark_key,
        Body=json.dumps(
            new_watermark_data,
            indent=2
        ).encode("utf-8"),
        ContentType="application/json"
    )

    print("Watermark updated:", new_watermark)

else:
    print("No new data found.")

Uploaded batch: orders/incremental/from=2026-10-27 03:28:00/to=2026-10-27 03:28:00/orders.csv
Rows uploaded: 1
Watermark updated: 2026-10-27 03:28:00


## Menguji incremental extration

### Menambahkan data dummy

In [193]:
test_insert_query = """
INSERT INTO orders (
    order_id,
    customer_id,
    product_id,
    qty,
    unit_price,
    status,
    order_ts,
    updated_at
)
VALUES (
    'MP7_INC_TEST_001',
    'C001',
    'P001',
    2,
    25000,
    'PAID',
    CURRENT_TIMESTAMP,
    CURRENT_TIMESTAMP
)
ON CONFLICT (order_id) DO NOTHING;
"""

cursor = conn.cursor()
cursor.execute(test_insert_query)
conn.commit()
cursor.close()

print("Test transaction inserted.")

Test transaction inserted.


In [194]:
# Ambil checkpoint terbaru
watermark_obj = s3.get_object(
    Bucket="toko-raw",
    Key=watermark_key
)

watermark_data = json.loads(
    watermark_obj["Body"].read().decode("utf-8")
)

last_watermark = watermark_data["last_updated_at"]

# Gunakan > karena kita ingin mengambil data yang benar-benar lebih baru
incremental_query = """
SELECT *
FROM orders
WHERE updated_at > %s
ORDER BY updated_at, order_id;
"""

incremental_df = pd.read_sql_query(
    incremental_query,
    conn,
    params=(last_watermark,)
)

print("Previous watermark:", last_watermark)
print("New incremental rows:", len(incremental_df))
print(incremental_df[["order_id", "status", "updated_at"]])

Previous watermark: 2026-10-27 03:28:00
New incremental rows: 1
           order_id status                 updated_at
0  MP7_INC_TEST_001   PAID 2026-11-20 13:50:25.288848


C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\4011143769.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  incremental_df = pd.read_sql_query(


In [195]:
if not incremental_df.empty:

    new_watermark = str(incremental_df["updated_at"].max())

    csv_bytes = incremental_df.to_csv(
        index=False,
        lineterminator="\n"
    ).encode("utf-8")

    batch_key = (
        f"orders/incremental/"
        f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )

    # Upload raw
    s3.put_object(
        Bucket="toko-raw",
        Key=batch_key,
        Body=csv_bytes,
        ContentType="text/csv"
    )

    # Verifikasi object benar-benar tersimpan
    uploaded_obj = s3.get_object(
        Bucket="toko-raw",
        Key=batch_key
    )

    uploaded_df = pd.read_csv(uploaded_obj["Body"])

    assert len(uploaded_df) == len(incremental_df)

    # Update checkpoint hanya setelah verifikasi berhasil
    new_watermark_data = {
        **watermark_data,
        "last_updated_at": new_watermark,
        "raw_key": batch_key,
        "updated_at": datetime.now().astimezone().isoformat()
    }

    s3.put_object(
        Bucket="toko-raw",
        Key=watermark_key,
        Body=json.dumps(
            new_watermark_data,
            indent=2
        ).encode("utf-8"),
        ContentType="application/json"
    )

    print("Uploaded:", batch_key)
    print("Verified rows:", len(uploaded_df))
    print("New watermark:", new_watermark)

else:
    print("No new data. Watermark not changed.")

Uploaded: orders/incremental/batch_20261120_221705.csv
Verified rows: 1
New watermark: 2026-11-20 13:50:25.288848


In [196]:
# Baca watermark terbaru dari MinIO
watermark_obj = s3.get_object(
    Bucket="toko-raw",
    Key=watermark_key
)

watermark_data = json.loads(
    watermark_obj["Body"].read().decode("utf-8")
)

last_watermark = watermark_data["last_updated_at"]

# Coba extract ulang
rerun_df = pd.read_sql_query(
    """
    SELECT *
    FROM orders
    WHERE updated_at > %s
    ORDER BY updated_at, order_id;
    """,
    conn,
    params=(last_watermark,)
)

print("Current watermark:", last_watermark)
print("Rows on rerun:", len(rerun_df))

Current watermark: 2026-11-20 13:50:25.288848
Rows on rerun: 0


C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\707106205.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rerun_df = pd.read_sql_query(


In [197]:
total_source_rows = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM orders",
    conn
).iloc[0]["total"]

watermark_data["row_count"] = int(total_source_rows)

s3.put_object(
    Bucket="toko-raw",
    Key=watermark_key,
    Body=json.dumps(
        watermark_data,
        indent=2
    ).encode("utf-8"),
    ContentType="application/json"
)

print("Checkpoint row_count:", total_source_rows)

Checkpoint row_count: 2001


C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\2654860440.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  total_source_rows = pd.read_sql_query(


# Idempotency

In [200]:
def upload_curated_idempotent(local_folder, bucket, prefix):
    local_folder = Path(local_folder)
    expected_keys = set()

    # Upload dengan object key yang konsisten
    for file_path in sorted(local_folder.rglob("*")):
        if not file_path.is_file():
            continue

        relative_path = file_path.relative_to(local_folder)

        if file_path.name == "_SUCCESS":
            object_key = f"{prefix}/_SUCCESS"

        elif file_path.suffix == ".parquet":
            parent = relative_path.parent.as_posix()

            if parent == ".":
                object_key = f"{prefix}/part-00000.parquet"
            else:
                object_key = f"{prefix}/{parent}/part-00000.parquet"
        else:
            continue

        s3.upload_file(
            str(file_path),
            bucket,
            object_key
        )

        expected_keys.add(object_key)

    # Hapus object lama yang memakai nama file UUID
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(
        Bucket=bucket,
        Prefix=f"{prefix}/"
    ):
        old_keys = [
            {"Key": obj["Key"]}
            for obj in page.get("Contents", [])
            if obj["Key"] not in expected_keys
        ]

        if old_keys:
            s3.delete_objects(
                Bucket=bucket,
                Delete={"Objects": old_keys}
            )

    print(f"Completed: {prefix}")
    print("Expected objects:", len(expected_keys))


upload_curated_idempotent(
    daily_local,
    "toko-curated",
    "daily_sales"
)

upload_curated_idempotent(
    product_local,
    "toko-curated",
    "product_sales"
)

Completed: daily_sales
Expected objects: 8
Completed: product_sales
Expected objects: 2


In [201]:
for prefix in ["daily_sales/", "product_sales/"]:

    response = s3.list_objects_v2(
        Bucket="toko-curated",
        Prefix=prefix
    )

    objects = [
        obj["Key"]
        for obj in response.get("Contents", [])
    ]

    print(f"\n{prefix}")
    print("Object count:", len(objects))

    for key in objects:
        print(key)


daily_sales/
Object count: 8
daily_sales/_SUCCESS
daily_sales/dt=2026-10-20/part-00000.parquet
daily_sales/dt=2026-10-21/part-00000.parquet
daily_sales/dt=2026-10-22/part-00000.parquet
daily_sales/dt=2026-10-23/part-00000.parquet
daily_sales/dt=2026-10-24/part-00000.parquet
daily_sales/dt=2026-10-25/part-00000.parquet
daily_sales/dt=2026-10-26/part-00000.parquet

product_sales/
Object count: 2
product_sales/_SUCCESS
product_sales/part-00000.parquet


# Laporan

### quality_report.csv

In [203]:
from pathlib import Path
import pandas as pd

outputs_dir = BASE_DIR / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

quality_report = pd.DataFrame([
    {"metric": "total_input_rows", "count": 2000},
    {"metric": "valid_rows", "count": 1269},
    {"metric": "rejected_rows", "count": 731},
    {"metric": "unsupported_status", "count": 724},
    {"metric": "invalid_quantity", "count": 10},
])

quality_report.to_csv(
    outputs_dir / "quality_report.csv",
    index=False
)

print(quality_report)
print("Saved:", outputs_dir / "quality_report.csv")

               metric  count
0    total_input_rows   2000
1          valid_rows   1269
2       rejected_rows    731
3  unsupported_status    724
4    invalid_quantity     10
Saved: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\outputs\quality_report.csv


### reconciliation.csv

In [204]:
reconciliation_report = pd.DataFrame([
    {
        "metric": "total_orders",
        "source_value": 1062,
        "lake_value": 1062
    },
    {
        "metric": "total_items",
        "source_value": 2623,
        "lake_value": 2623
    },
    {
        "metric": "total_revenue",
        "source_value": 274070000,
        "lake_value": 274070000
    },
    {
        "metric": "rejected_rows",
        "source_value": 731,
        "lake_value": 731
    }
])

reconciliation_report["diff"] = (
    reconciliation_report["source_value"]
    - reconciliation_report["lake_value"]
)

reconciliation_report["status"] = (
    reconciliation_report["diff"]
    .apply(lambda x: "PASS" if x == 0 else "FAIL")
)

reconciliation_report.to_csv(
    outputs_dir / "reconciliation.csv",
    index=False
)

print(reconciliation_report)
print("Saved:", outputs_dir / "reconciliation.csv")

          metric  source_value  lake_value  diff status
0   total_orders          1062        1062     0   PASS
1    total_items          2623        2623     0   PASS
2  total_revenue     274070000   274070000     0   PASS
3  rejected_rows           731         731     0   PASS
Saved: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\outputs\reconciliation.csv


### failute test

In [205]:
failure_test_content = """
# Failure Test - Toko Bandung

## Scenario 1: Process stops after raw upload

### Gejala
File raw sudah tersimpan di MinIO, tetapi watermark belum diperbarui.

### Penyebab
Proses gagal sebelum checkpoint berhasil ditulis.

### Recovery
1. Baca watermark terakhir dari MinIO.
2. Jalankan kembali incremental extraction.
3. Upload batch dengan object key yang konsisten.
4. Verifikasi jumlah baris dan checksum.
5. Update watermark setelah raw berhasil diverifikasi.

### Pencegahan
Watermark hanya diperbarui setelah raw berhasil ditulis dan diverifikasi.

Status: Recovery procedure documented, not physically simulated.

## Scenario 2: Incremental rerun

### Gejala
Pipeline dijalankan ulang setelah watermark diperbarui.

### Hasil pengujian
- Transaksi baru berhasil diambil: 1 baris.
- Raw berhasil di-upload dan diverifikasi.
- Watermark berhasil diperbarui.
- Incremental rows pada rerun: 0.

### Pencegahan
Menggunakan updated_at sebagai watermark dan order_id sebagai kunci deduplikasi.

Status: Tested.

## Scenario 3: Duplicate Curated upload

### Gejala
Pipeline dijalankan ulang dan berpotensi menghasilkan file Parquet baru.

### Recovery
Gunakan object key konsisten untuk menggantikan file sebelumnya.

### Hasil pengujian
- daily_sales: 8 objects.
- product_sales: 2 objects.
- Object key menggunakan nama deterministik.

### Pencegahan
Upload menggunakan deterministic object key.

Status: Object structure verified. Confirm object counts remain unchanged after a second upload.
"""

failure_path = outputs_dir / "failure_test.md"
failure_path.write_text(failure_test_content.strip(), encoding="utf-8")

print("Saved:", failure_path)

Saved: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\outputs\failure_test.md


### dokumentasi proyek

In [211]:

from pathlib import Path

docs_dir = BASE_DIR / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

documents = {
    "DATA_CONTRACT.md": """
# Data Contract - Toko Bandung

## 1. Informasi Sumber
- Source system: PostgreSQL
- Database: toko
- Source table: orders
- Data owner: Toko Bandung (simulasi)
- Ingestion method: Initial load dan incremental extraction
- Watermark column: updated_at

## 2. Schema dan Aturan Kolom

| Kolom | Arti | Aturan |
|---|---|---|
| order_id | ID transaksi | Tidak boleh kosong, primary key |
| customer_id | ID pelanggan | Identitas pelanggan |
| product_id | ID produk | Tidak boleh kosong |
| qty | Jumlah barang | Harus lebih besar dari 0 |
| unit_price | Harga satuan | 25000, 50000, 75000, 120000, 250000 |
| status | Status transaksi | PAID atau CANCELLED |
| order_ts | Waktu transaksi | Tidak boleh kosong |
| updated_at | Waktu perubahan data | Digunakan sebagai watermark |

## 3. Primary Key dan Deduplication
- Primary key: order_id
- Duplikasi ditangani menggunakan Window Function Spark.
- Record dengan updated_at terbaru diprioritaskan.

## 4. Aturan Status
Status distandardisasi dengan TRIM dan UPPER.

Status yang diterima:
- PAID
- CANCELLED

Status lain, misalnya SHIPPED dan PAYED, masuk quarantine.

## 5. Aturan Bisnis
- qty harus lebih besar dari 0.
- unit_price harus termasuk daftar harga yang diizinkan.
- order_id dan product_id tidak boleh kosong.
- order_ts tidak boleh kosong.
- Transaksi CANCELLED tidak dihitung dalam agregasi penjualan.

## 6. Penanganan Data Tidak Valid
Baris yang gagal validasi tidak masuk ke processed valid data dan curated.
Baris tersebut disimpan di quarantine beserta reject_reason.

## 7. Contoh
Valid:
order_id=ORD001, product_id=P001, qty=2, unit_price=25000, status=PAID

Tidak valid:
order_id=ORD002, product_id=P001, qty=0, unit_price=25000, status=PAID

Alasan: Invalid quantity.
""",

    "ARCHITECTURE.md": """
# Architecture - Toko Bandung

## 1. Overview
Pipeline menggunakan PostgreSQL sebagai source database,
MinIO sebagai data lake, dan PySpark untuk transformasi.

## 2. Data Flow

```text
PostgreSQL (orders)
       |
       | Initial / Incremental Extraction
       | updated_at watermark
       v
MinIO - toko-raw
       |
       | Download raw object to local temporary file
       v
PySpark - Standardization & Quality Gate
       |
       +--------------------+
       |                    |
       v                    v
MinIO - toko-processed   MinIO - quarantine
       |
       | Aggregation
       v
MinIO - toko-curated
       |
       +-----------------------+
       |                       |
       v                       v
   daily_sales             product_sales
```

## 3. Data Lake Zones

### Raw
Menyimpan data sumber dari PostgreSQL, file awal, metadata, dan checkpoint watermark.

### Processed
Menyimpan data yang telah distandardisasi dan lolos quality gate dalam format Parquet.

### Quarantine
Menyimpan data yang gagal validasi beserta alasan penolakannya.

### Curated
Menyimpan hasil agregasi untuk kebutuhan analisis:
- daily_sales
- product_sales

## 4. Incremental Processing
Data baru diambil berdasarkan kolom updated_at.
Watermark diperbarui setelah data berhasil diunggah dan diverifikasi.

## 5. Catatan Implementasi
Spark membaca file yang diunduh sementara dari MinIO menggunakan boto3.
File hasil transformasi kemudian diunggah kembali ke MinIO.
""",

    "RUNBOOK.md": """
# Runbook - Toko Bandung

## 1. Prasyarat
- Docker Desktop
- Python
- Java
- PostgreSQL
- MinIO
- Dependencies dari requirements.txt

## 2. Menjalankan Infrastruktur
Jalankan container menggunakan Docker Compose:

```bash
docker compose up -d
```

Pastikan PostgreSQL dan MinIO berjalan.

## 3. Menjalankan Pipeline
1. Aktifkan virtual environment.
2. Pastikan konfigurasi environment tersedia di file .env.
3. Buka notebook notebooks/minio_exploration.ipynb.
4. Jalankan cell secara berurutan.
5. Periksa hasil pada bucket MinIO.

## 4. Pemeriksaan Hasil
Periksa keberadaan data pada:
- toko-raw
- toko-processed
- toko-curated

Pastikan hasil processed dan curated memiliki file _SUCCESS.

## 5. Incremental Load
1. Baca watermark terakhir dari checkpoint.
2. Ambil data PostgreSQL dengan updated_at lebih besar dari watermark.
3. Unggah hasil extract ke MinIO Raw.
4. Verifikasi jumlah baris dan checksum.
5. Perbarui watermark setelah verifikasi berhasil.

## 6. Recovery dan Failure Handling

### Raw berhasil diunggah tetapi watermark belum diperbarui
Jalankan kembali proses verifikasi raw.
Jika data valid, lanjutkan pembaruan watermark.
Jangan langsung menghapus raw object.

### Incremental dijalankan ulang
Gunakan watermark terakhir agar data yang sama tidak diekstrak kembali.

### Curated diunggah ulang
Gunakan object key deterministik agar hasil rerun tidak menghasilkan file duplikat.

## 7. Quality Gate
Data yang tidak memenuhi Data Contract diarahkan ke quarantine.
Periksa reject_reason untuk mengetahui penyebab kegagalan.
""",

    "README.md": """
# Mini Project 7 - Pipeline Toko Bandung

## Deskripsi
Proyek ini membangun pipeline data dari PostgreSQL ke MinIO Data Lake
menggunakan PySpark untuk proses standardisasi, validasi, dan agregasi.

## Arsitektur
PostgreSQL -> MinIO Raw -> PySpark -> Processed / Quarantine -> Curated

## Teknologi
- PostgreSQL
- MinIO
- PySpark
- Python
- Docker
- Jupyter Notebook

## Struktur Project

```text
mp7_pipeline_toko_bandung/
├── data/
├── docs/
│   ├── DATA_CONTRACT.md
│   ├── ARCHITECTURE.md
│   └── RUNBOOK.md
├── notebooks/
│   └── minio_exploration.ipynb
├── outputs/
├── docker-compose.yml
├── requirements.txt
├── .env
└── README.md
```

## Data Lake Zones
- toko-raw: data sumber, metadata, dan checkpoint.
- toko-processed: data yang telah distandardisasi dan lolos validasi.
- toko-processed/quarantine: data yang gagal validasi.
- toko-curated: hasil agregasi daily_sales dan product_sales.

## Hasil Quality Gate
- Total input awal: 2.000 baris
- Valid rows: 1.269 baris
- Rejected rows: 731 baris

## Hasil Curated
- Total paid orders: 1.062
- Total items: 2.623
- Total revenue: Rp274.070.000
- Daily sales: 7 hari
- Product sales: 40 produk

## Incremental Load
Incremental extraction menggunakan updated_at sebagai watermark.
Pengujian menghasilkan satu record baru dan rerun berikutnya menghasilkan nol record baru.

## Idempotency
Object key Curated menggunakan nama deterministik.
Dengan strategi ini, rerun dapat menggantikan hasil sebelumnya tanpa menambah object duplikat.

## Dokumentasi
- [Data Contract](docs/DATA_CONTRACT.md)
- [Architecture](docs/ARCHITECTURE.md)
- [Runbook](docs/RUNBOOK.md)

## Keamanan
File .env dan kredensial tidak boleh dimasukkan ke Git.
"""
}

for filename, content in documents.items():
    file_path = (
        BASE_DIR / filename
        if filename == "README.md"
        else docs_dir / filename
    )
    file_path.write_text(content.strip() + "\n", encoding="utf-8")
    print(f"Created: {file_path}")

print("\nSemua dokumentasi berhasil dibuat.")


Created: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\docs\DATA_CONTRACT.md
Created: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\docs\ARCHITECTURE.md
Created: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\docs\RUNBOOK.md
Created: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\README.md

Semua dokumentasi berhasil dibuat.


# Benchmark Spark

In [212]:

import time
import pandas as pd
from pyspark.sql import functions as F

benchmark_results = []

# Jumlah input dihitung sekali agar tidak berulang
input_rows = processed_df.count()

# Uji tiga konfigurasi shuffle partitions
for partition_count in [2, 8, 32]:

    spark.conf.set(
        "spark.sql.shuffle.partitions",
        str(partition_count)
    )

    # Warm-up: menjalankan query sebelum pengukuran
    (
        processed_df
        .groupBy("product_id")
        .agg(F.sum("qty").alias("total_qty"))
        .count()
    )

    # Tiga kali pengukuran
    for run_no in range(1, 4):

        start_time = time.perf_counter()

        result_df = (
            processed_df
            .groupBy("product_id")
            .agg(
                F.sum("qty").alias("total_qty"),
                F.sum(
                    F.col("qty") * F.col("unit_price")
                ).alias("revenue")
            )
        )

        output_rows = result_df.count()

        elapsed = time.perf_counter() - start_time

        benchmark_results.append({
            "scenario": f"shuffle_{partition_count}",
            "run_no": run_no,
            "elapsed_sec": round(elapsed, 4),
            "input_rows": input_rows,
            "output_rows": output_rows,
            "partitions": partition_count,
            "output_bytes": 0
        })

# Simpan hasil benchmark
benchmark_df = pd.DataFrame(benchmark_results)

benchmark_path = outputs_dir / "partition_benchmark.csv"
benchmark_df.to_csv(benchmark_path, index=False)

print("Benchmark selesai!")
print(benchmark_df.to_string(index=False))
print(f"\nFile tersimpan: {benchmark_path}")


Benchmark selesai!
  scenario  run_no  elapsed_sec  input_rows  output_rows  partitions  output_bytes
 shuffle_2       1       0.1725        1269           40           2             0
 shuffle_2       2       0.1853        1269           40           2             0
 shuffle_2       3       0.1856        1269           40           2             0
 shuffle_8       1       0.1680        1269           40           8             0
 shuffle_8       2       0.2077        1269           40           8             0
 shuffle_8       3       0.1871        1269           40           8             0
shuffle_32       1       0.2219        1269           40          32             0
shuffle_32       2       0.2198        1269           40          32             0
shuffle_32       3       0.2171        1269           40          32             0

File tersimpan: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\outputs\partition_benchmark.csv


## hitung median

In [213]:

median_df = (
    benchmark_df
    .groupby("scenario", as_index=False)
    .agg(
        median_elapsed_sec=("elapsed_sec", "median"),
        input_rows=("input_rows", "first"),
        output_rows=("output_rows", "first"),
        partitions=("partitions", "first")
    )
    .sort_values("partitions")
)

print("HASIL PERBANDINGAN MEDIAN")
print(median_df.to_string(index=False))

median_path = outputs_dir / "partition_benchmark_median.csv"
median_df.to_csv(median_path, index=False)

print(f"\nMedian tersimpan: {median_path}")


HASIL PERBANDINGAN MEDIAN
  scenario  median_elapsed_sec  input_rows  output_rows  partitions
 shuffle_2              0.1853        1269           40           2
 shuffle_8              0.1871        1269           40           8
shuffle_32              0.2198        1269           40          32

Median tersimpan: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\outputs\partition_benchmark_median.csv


In [214]:

benchmark_report = """
# Spark Optimization Experiment

## 1. Tujuan
Membandingkan performa Spark menggunakan konfigurasi
spark.sql.shuffle.partitions yang berbeda.

## 2. Metodologi
- Dataset: Processed orders
- Input rows: 1.269
- Operasi: Group By product_id dengan agregasi total quantity dan revenue
- Konfigurasi: 2, 8, dan 32 shuffle partitions
- Pengukuran: 3 kali untuk setiap konfigurasi
- Warm-up dilakukan sebelum pengukuran.
- Perbandingan menggunakan median elapsed time.

## 3. Hasil Benchmark

| Scenario | Median (seconds) | Input Rows | Output Rows |
|---|---:|---:|---:|
| shuffle_2 | 0.1853 | 1269 | 40 |
| shuffle_8 | 0.1871 | 1269 | 40 |
| shuffle_32 | 0.2198 | 1269 | 40 |

## 4. Analisis
Konfigurasi shuffle_2 menghasilkan median waktu tercepat,
yaitu 0.1853 detik.

Konfigurasi shuffle_8 memiliki performa yang hampir sama,
dengan median 0.1871 detik.

Konfigurasi shuffle_32 menghasilkan waktu paling lambat,
yaitu 0.2198 detik.

Hal ini menunjukkan bahwa penambahan jumlah shuffle partitions
tidak selalu meningkatkan performa. Pada dataset yang relatif
kecil, overhead pengelolaan partisi tambahan dapat mengurangi
keuntungan paralelisme.

## 5. Kesimpulan
Berdasarkan eksperimen ini, shuffle_2 menjadi konfigurasi
tercepat untuk dataset dan lingkungan pengujian yang digunakan.

Hasil ini tidak dapat digeneralisasi untuk semua ukuran dataset.
Pengujian tambahan diperlukan untuk menentukan konfigurasi
optimal pada volume data yang lebih besar.

## 6. Catatan
Eksperimen mengukur waktu eksekusi agregasi tanpa menulis
hasil ke Parquet. Karena itu, output_bytes dicatat sebagai 0
dan bukan ukuran file hasil.
"""

report_path = docs_dir / "SPARK_BENCHMARK.md"
report_path.write_text(
    benchmark_report.strip() + "\n",
    encoding="utf-8"
)

print(f"Benchmark report created: {report_path}")


Benchmark report created: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\docs\SPARK_BENCHMARK.md
